# YOLO Detection Example

## YOLO Object Detection

This notebook is an example of object detection with **Ultralytics YOLO**.

```
Image (URL or file) → YOLO detector → Bounding boxes + class labels + confidence scores
```

Unlike classification (one label per image), detection **locates multiple objects** within an image and draws a bounding box around each one.

### Available models

| Model | Params | Speed |
|-------|--------|-------|
| `yolo11x.pt` | 56.9 M | slowest / most accurate |
| `yolo11l.pt` | 25.3 M | — |
| `yolo11m.pt` | 20.1 M | — |
| `yolo11s.pt` |  9.4 M | — |
| `yolo11n.pt` |  2.6 M | fastest / lightest |

Change `MODEL_NAME` in the model loading cell to switch between them.

📄 [Ultralytics YOLO docs](https://docs.ultralytics.com/)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH="/content/drive/MyDrive/Practical-ML/yolo"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls

In [ ]:
# Ultralytics installation
!pip install ultralytics -q

import ultralytics
ultralytics.checks()  # check environment

In [ ]:
# Download sample images from GitHub
import os

# Download only the images folder from GitHub (shallow sparse checkout)
if not os.path.exists(f'{PROJECT_PATH}/images'):
    !git clone --depth=1 --filter=blob:none --sparse https://github.com/mastnk/practical-ml /tmp/practical-ml -q
    !cd /tmp/practical-ml && git sparse-checkout set yolo/images
    !cp -r /tmp/practical-ml/yolo/images "{PROJECT_PATH}/images"
    print('Images downloaded.')
else:
    print('Images already exist, skipping.')

%cd "{PROJECT_PATH}"
!ls images/

## Using Your Own Images

There are three ways to provide images:

**① Change the GitHub download path**  
Edit the `git sparse-checkout set` line in the download cell to point to a different folder in the repository:
```python
!cd /tmp/practical-ml && git sparse-checkout set yolo/images  # ← change this path
```

**② Upload images directly in Colab**  
Open the file browser on the left sidebar, navigate to the `images/` folder, and drag & drop your files.  
Or use the following code in a new cell:
```python
from google.colab import files
uploaded = files.upload()  # opens a file picker dialog
# move uploaded files to the images folder
import shutil, os
for fname in uploaded:
    shutil.move(fname, f'{PROJECT_PATH}/images/{fname}')
```

**③ Use images already saved in Google Drive**  
Copy or move your image files into `PROJECT_PATH/images/` via the Drive web interface, then re-run the classification/detection cell.

In [ ]:
# YOLO detection model loading
from ultralytics import YOLO

MODEL_NAME = 'yolo11x.pt'  # 56.9M params
MODEL_NAME = 'yolo11l.pt'  # 25.3M params
MODEL_NAME = 'yolo11m.pt'  # 20.1M params
MODEL_NAME = 'yolo11s.pt'  #  9.4M params
MODEL_NAME = 'yolo11n.pt'  #  2.6M params

model = YOLO(MODEL_NAME)

## Selecting a Model

In the cell below, multiple `MODEL_NAME = ...` lines are listed — **only the last assignment takes effect**.  
To switch models, move your preferred line to the bottom, or comment out all others:

```python
# MODEL_NAME = 'yolo11x.pt'  # 56.9M params — most accurate, slowest
# MODEL_NAME = 'yolo11l.pt'  # 25.3M params
# MODEL_NAME = 'yolo11m.pt'  # 20.1M params
# MODEL_NAME = 'yolo11s.pt'  #  9.4M params
MODEL_NAME   = 'yolo11n.pt'  #  2.6M params — fastest, lightest  ← active
```

Larger models are more accurate but take longer to run. Start with `yolo11n.pt` for quick experiments.

In [ ]:
from PIL import Image
import requests
from io import BytesIO
from IPython.display import display

def detect_url(url: str, conf: float = 0.25):
    image   = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
    results = model.predict(image, conf=conf, verbose=False)

    annotated = Image.fromarray(results[0].plot()[:, :, ::-1])  # BGR -> RGB
    display(annotated)

    boxes = results[0].boxes
    names = results[0].names
    for box in boxes:
        cls_id   = int(box.cls[0])
        conf_val = float(box.conf[0])
        print(f'  {names[cls_id]:<20} {conf_val*100:.1f}%')

    return results


def detect_file(path: str, conf: float = 0.25):
    image   = Image.open(path).convert('RGB')
    results = model.predict(image, conf=conf, verbose=False)

    annotated = Image.fromarray(results[0].plot()[:, :, ::-1])  # BGR -> RGB
    display(annotated)

    boxes = results[0].boxes
    names = results[0].names
    for box in boxes:
        cls_id   = int(box.cls[0])
        conf_val = float(box.conf[0])
        print(f'  {names[cls_id]:<20} {conf_val*100:.1f}%')

    return results

In [ ]:
# Example: detect from URL
url = 'https://cdn.pixabay.com/photo/2016/12/13/05/15/puppy-1903313_640.jpg'
results = detect_url(url)

In [ ]:
# Detect objects in all downloaded sample images
import glob

image_dir = f'{PROJECT_PATH}/images'
exts = ['jpg', 'jpeg', 'png', 'bmp', 'JPG', 'JPEG', 'PNG', 'BMP']

filepaths = []
for ext in exts:
    filepaths += sorted(glob.glob(f'{image_dir}/*.{ext}'))

for path in filepaths:
    print(f'--- {os.path.basename(path)} ---')
    detect_file(path)
    print()

💾 **Don't forget to save this notebook!**

To keep your work, go to File → Save a copy in Drive before closing.

[![Return to GitHub](https://img.shields.io/badge/GitHub-Return-black?logo=github)](https://github.com/mastnk/practical-ml)